# FIFA Match Outcome Predictor — Analysis Notebook

This notebook covers:
1. Exploratory Data Analysis (EDA)
2. Elo Rating Computation
3. Feature Engineering
4. Model Training & Evaluation
5. Interactive Predictions

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded.')

## 1. Load Data

In [ ]:
from src.data_loader import load_all
df = load_all()
print(f'Dataset shape: {df.shape}')
df.head()

## 2. EDA — Match Distribution

In [ ]:
# Outcome distribution
label_map = {0: 'Away Win', 1: 'Draw', 2: 'Home Win'}
outcome_counts = df['outcome'].map(label_map).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(outcome_counts.values, labels=outcome_counts.index,
            autopct='%1.1f%%', colors=['#ef4444', '#f59e0b', '#3b82f6'])
axes[0].set_title('Match Outcome Distribution', fontsize=14)

# Matches per year
df['year'] = df['date'].dt.year
matches_per_year = df.groupby('year').size()
axes[1].bar(matches_per_year.index, matches_per_year.values, color='#3b82f6', alpha=0.8)
axes[1].set_title('Number of Matches per Year', fontsize=14)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Matches')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most active teams
all_teams = pd.concat([
    df['home_team'], df['away_team']
]).value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
all_teams.plot(kind='bar', ax=ax, color='#7c3aed', alpha=0.85)
ax.set_title('Top 15 Most Active International Teams', fontsize=14)
ax.set_ylabel('Number of Matches')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Goals distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['home_score'], bins=range(0, 15), alpha=0.7, color='#3b82f6', label='Home')
axes[0].hist(df['away_score'], bins=range(0, 15), alpha=0.7, color='#ef4444', label='Away')
axes[0].set_title('Goals Distribution')
axes[0].set_xlabel('Goals')
axes[0].legend()

# Home vs Away win rate over decades
df['decade'] = (df['year'] // 10) * 10
home_wr = df.groupby('decade').apply(lambda x: (x['outcome'] == 2).mean())
away_wr = df.groupby('decade').apply(lambda x: (x['outcome'] == 0).mean())
draw_r  = df.groupby('decade').apply(lambda x: (x['outcome'] == 1).mean())

axes[1].plot(home_wr.index, home_wr.values, 'o-', color='#3b82f6', label='Home Win %')
axes[1].plot(away_wr.index, away_wr.values, 's-', color='#ef4444', label='Away Win %')
axes[1].plot(draw_r.index,  draw_r.values,  '^-', color='#f59e0b', label='Draw %')
axes[1].set_title('Win/Draw Rates by Decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Elo Rating System

In [ ]:
from src.elo_calculator import EloSystem

elo_system = EloSystem()
df = elo_system.calculate(df)

print('\nTop 20 Teams by Elo Rating:')
elo_system.top_teams(20)

In [ ]:
# Track Elo over time for Brazil and Argentina
teams_to_track = ['Brazil', 'Germany', 'France', 'Argentina', 'England']

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#3b82f6', '#f59e0b', '#10b981', '#ef4444', '#8b5cf6']

for team, color in zip(teams_to_track, colors):
    home_mask = df['home_team'] == team
    away_mask = df['away_team'] == team
    
    team_elo = pd.concat([
        df[home_mask][['date', 'home_elo']].rename(columns={'home_elo': 'elo'}),
        df[away_mask][['date', 'away_elo']].rename(columns={'away_elo': 'elo'})
    ]).sort_values('date')
    
    ax.plot(team_elo['date'], team_elo['elo'], label=team, color=color, linewidth=1.5, alpha=0.85)

ax.set_title('Elo Rating Over Time — Top Nations', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Elo Rating')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import build_features, get_feature_columns

# Note: This takes a few minutes on the full dataset
# Use a subset for quick exploration:
df_sample = df.tail(10000).copy()  # last 10k matches for speed
df_feats = build_features(df_sample, verbose=True)

FEATURE_COLS = get_feature_columns()
df_feats[FEATURE_COLS + ['outcome']].describe()

In [ ]:
# Correlation heatmap
corr_data = df_feats[FEATURE_COLS + ['outcome']].dropna()
corr = corr_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8},
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation

Run `python src/train.py` for full training. Here we do a quick demo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

df_model = df_feats[FEATURE_COLS + ['outcome']].dropna()
X = df_model[FEATURE_COLS].values
y = df_model['outcome'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Away Win', 'Draw', 'Home Win']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Away Win', 'Draw', 'Home Win'],
    yticklabels=['Away Win', 'Draw', 'Home Win'],
    ax=ax
)
ax.set_title('Logistic Regression — Confusion Matrix')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Interactive Prediction

Make sure `python src/train.py` has been run first.

In [ ]:
try:
    from src.predict import Predictor
    predictor = Predictor()
    
    # Try a match
    result = predictor.predict('Brazil', 'Argentina', neutral=True, tournament='FIFA World Cup')
    predictor.print_prediction(result)
except FileNotFoundError:
    print('Model not found. Run: python src/train.py')

In [ ]:
# Test multiple matchups
try:
    matchups = [
        ('Brazil', 'Argentina'),
        ('Germany', 'France'),
        ('England', 'Spain'),
        ('Portugal', 'Belgium'),
    ]
    
    for ht, at in matchups:
        r = predictor.predict(ht, at, neutral=True, tournament='FIFA World Cup')
        print(f'{ht:>12} vs {at:<12} | {ht} Win: {r["home_win_prob"]*100:5.1f}%  '
              f'Draw: {r["draw_prob"]*100:5.1f}%  {at} Win: {r["away_win_prob"]*100:5.1f}%')
except:
    print('Run: python src/train.py first')